In [25]:
from datasets import load_dataset
import matplotlib as plt
import pandas as pd

In [ ]:
train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")
test = load_dataset("amarfurt/art-nlp-hs26", split="test")

print(len(train), len(valid), len(test))

16801 1000 1532


In [3]:
print(train)

Dataset({
    features: ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2', 'label'],
    num_rows: 16801
})


In [7]:
train_df = train.to_pandas()
valid_df = valid.to_pandas()
test_df = test.to_pandas()

print(len(train_df), len(valid_df), len(test_df)) #check if we stil have the same row count

16801 1000 1532


In [6]:
train_df.head(10)

,observation_1,observation_2,hypothesis_1,hypothesis_2,label
0,A baby panda was lost in the woods.,"He was reunited with his parents, who also liv...",He was never lost.,All the people from the zoo looked for him.,2
1,A bank teller told me to rob the bank where sh...,We split the money equally.,The mountain told me when no one would be there.,The teller told me when no one would be there.,2
2,A bee was buzzing around my head.,I didn't bother to explain the situation.,It was because of the honey in my hair.,It was because of the blood in my stool.,1
3,A blizzard suddenly started in the winter.,So the neighbor gave the boy some cash.,A boy offered to shovel his neighbors driveway.,A boy never offered to mow his neighbors drive...,1
4,A boy was outside playing when he saw a balloo...,Once he held the balloon he popped it with a s...,The boy popped the balloon.,The boy caught the ballon.,2
5,A farmer planted some corn seed in the ground ...,He sold a lot of corn at the market that day.,The farmer haddn't harvested any corn.,The farmer had already harvested a bunch of corn.,2
6,"A farmer woke up one morning, and found his pi...","It turned out, the pigs had been left in the s...",The dogs were not where they should have been.,The pigs were not where they should have been.,2
7,A fisherman living in a jungle noticed dark ra...,"He woke up later in his village, safe from the...",The fisherman's boat sunk.,The fisherman was rescued by a nearby boat.,2
8,A game was brought back from vacation.,The games headsets were all destroyed.,The suitcases got thrown around by airport staff.,The suitcases were not thrown around by airpor...,1
9,A janitor was called to a classroom to change ...,He put his ladder away and left the classroom.,The janitor slipped off the ladder and hurt hi...,The janitor unfolded a ladder and changed the ...,2


In [ ]:
#Class Balance train
h_1 = len(train_df[train_df["label"] == 1])
h_2 = len(train_df[train_df["label"] == 2])

print(f"Count h1: {h_1}")
print(f"Count h2: {h_2}")
print(f"Ration of h1 to h2: {round(h_1 / h_2, 2)}")


Count h1: 8461
Count h2: 8340
Ration of h1 to h2: 1.01


In [19]:
#Class Balance validation
h_1 = len(valid_df[valid_df["label"] == 1])
h_2 = len(valid_df[valid_df["label"] == 2])

print(f"Count h1: {h_1}")
print(f"Count h2: {h_2}")
print(f"Ration of h1 to h2: {round(h_1 / h_2, 2)}")


Count h1: 508
Count h2: 492
Ration of h1 to h2: 1.03


In [20]:
#Class Balance test
h_1 = len(test_df[test_df["label"] == 1])
h_2 = len(test_df[test_df["label"] == 2])

print(f"Count h1: {h_1}")
print(f"Count h2: {h_2}")
print(f"Ration of h1 to h2: {round(h_1 / h_2, 2)}")


Count h1: 781
Count h2: 751
Ration of h1 to h2: 1.04


In [23]:
#Sequence lengths
text_cols = [
    "observation_1",
    "observation_2",
    "hypothesis_1",
    "hypothesis_2"
]

for col in text_cols:
    train_df[f"{col}_length"] = train_df[col].str.split().str.len()
    
length_cols = [f"{col}_length" for col in text_cols]

train_df[length_cols].describe().round(2)

,observation_1_length,observation_2_length,hypothesis_1_length,hypothesis_2_length
count,16801.00,16801.00,16801.00,16801.00
mean,8.08,9.28,8.37,8.39
std,2.53,2.73,3.00,3.03
min,2.00,2.00,1.00,1.00
25%,6.00,7.00,6.00,6.00
50%,8.00,9.00,8.00,8.00
75%,10.00,11.00,10.00,10.00
max,16.00,19.00,30.00,30.00


In [26]:
all_lengths = pd.concat(
    [train_df[col] for col in length_cols],
    ignore_index=True
)

print(all_lengths.describe())
print("95th percentile:", all_lengths.quantile(0.95))
print("99th percentile:", all_lengths.quantile(0.99))

count    67204.000000
mean         8.531248
std          2.865966
min          1.000000
25%          6.000000
50%          8.000000
75%         10.000000
max         30.000000
dtype: float64
95th percentile: 13.0
99th percentile: 17.0


In [27]:
for col in text_cols:
    print(f"\n{col}:")
    display(train_df.loc[train_df[f"{col}_length"] == 1, [col]])


observation_1:


,observation_1



observation_2:


,observation_2



hypothesis_1:


,hypothesis_1
1671,shesaid.
6855,{}.
8765,{}.
8768,.



hypothesis_2:


,hypothesis_2
2260,{}.
4463,NONSENSICAL.
7734,.
11355,{}.


In [29]:
long_df = train_df[text_cols].melt(
    var_name="source",
    value_name="text"
)

long_df["length"] = long_df["text"].str.split().str.len()

# Shortest sentences
display(long_df.nsmallest(10, "length"))

# Longest sentences
display(long_df.nlargest(10, "length"))

,source,text,length
35273,hypothesis_1,shesaid.,1
40457,hypothesis_1,{}.,1
42367,hypothesis_1,{}.,1
42370,hypothesis_1,.,1
52663,hypothesis_2,{}.,1
54866,hypothesis_2,NONSENSICAL.,1
58137,hypothesis_2,.,1
61758,hypothesis_2,{}.,1
8628,observation_1,Strike two!,2
19233,observation_2,They burned!,2


,source,text,length
42470,hypothesis_1,Amy needed to travel to Glastonbury for a fest...,30
59271,hypothesis_2,Amy needed to travel to Glastonbury for a fest...,30
34754,hypothesis_1,"Anna's parents loved their children, and playe...",29
40065,hypothesis_1,A boy knocked on the door selling candy. After...,29
43275,hypothesis_1,When same woke up he didn't realize he had fal...,29
44568,hypothesis_1,Abbey planned to use it to record her father w...,29
49656,hypothesis_1,Nate asked Linda to pick her up at 7:30 at her...,29
51555,hypothesis_2,Anna's parents loved their children and played...,29
55142,hypothesis_2,I found that in back I had a tomato plant that...,29
60076,hypothesis_2,When Sam woke up he didn't realize he had fall...,29
